[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_diffsynth_practive/blob/main/anima_minimal_colab.ipynb)

# Minimal Anima T2I — extracted DiffSynth path

ComfyUI / Diffusers / DiffSynth를 설치하거나 별도 git clone하지 않습니다. 이 저장소의 최소 모델 정의와 일반 Python 라이브러리만 사용합니다. 모델 가중치는 공개 Hugging Face `circlestone-labs/Anima`에서 인증 없이 받습니다.


In [ ]:
# 1. Colab 기본 torch / transformers / huggingface_hub / Pillow는 그대로 사용
!pip -q install safetensors einops sentencepiece tqdm


In [ ]:
# 2. 이 실습 저장소만 clone / 최신 코드 강제 반영
from pathlib import Path
import importlib, subprocess, sys, torch

REPO = 'https://github.com/HisameOgasahara/tmp_diffsynth_practive.git'
ROOT = Path('/content/tmp_diffsynth_practive')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO, str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'reset', '--hard', 'origin/main'], check=True)

SRC = str(ROOT / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

for name in list(sys.modules):
    if name == 'anima_core' or name.startswith('anima_core.'):
        del sys.modules[name]
importlib.invalidate_caches()

commit = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('commit:', commit)
print(torch.__version__, torch.cuda.get_device_name() if torch.cuda.is_available() else 'NO CUDA')


In [ ]:
# 3. Baseline 생성값 — 수정 후 JSON으로 저장
import json
from pathlib import Path

PROMPT = '@ogipote, 1girl, plana (blue archive), upper body, \n, solo, cute, cheerful, innocent, expressionless, looking_at_viewer, alternte costume, white school uniform, blue_ribbon, white_socks, school_theme, sitting, holding_tablet, sunlight, soft_lighting, gentle_breeze, floating_particles, futuristic_city, academy_city, glass_building, blue_and_white_theme, pastel_colors, vibrant_colors, depth_of_field, wallpaper' # @param {type:'string'}
NEGATIVE_PROMPT = 'worst quality, low quality, score_1, score_2, score_3, blurry, jpeg artifacts' # @param {type:'string'}
SEED = 1113280783077040 # @param {type:'integer'}
STEPS = 30 # @param {type:'integer'}
CFG = 4.0 # @param {type:'number'}
WIDTH = 1216 # @param {type:'integer'}
HEIGHT = 832 # @param {type:'integer'}
DENOISE = 1.0 # @param {type:'number'}
SHIFT = 3.0 # @param {type:'number'}

CONFIG_PATH = Path('/content/anima_minimal_config.json')
config = {
    'prompt': PROMPT,
    'negative_prompt': NEGATIVE_PROMPT,
    'seed': int(SEED),
    'steps': int(STEPS),
    'cfg': float(CFG),
    'width': int(WIDTH),
    'height': int(HEIGHT),
    'denoise': float(DENOISE),
    'shift': float(SHIFT),
    'device': 'cuda',
    'dtype': 'float16',
}
CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
print(CONFIG_PATH)
print(json.dumps(config, ensure_ascii=False, indent=2))


In [ ]:
# 4. 공개 HF에서 Anima 가중치 미리 받기 — 선택 셀, token/auth 없음
from anima_core.runtime import download_weights
download_weights()
print('가중치 캐시 준비 완료')


In [ ]:
# 5. Text Encoder -> positive / negative conditioning 저장
import gc, json, torch
from pathlib import Path
from anima_core.runtime import download_weights, load_text_encoder, load_tokenizers, encode_prompt

CONFIG_PATH = Path('/content/anima_minimal_config.json')
if not CONFIG_PATH.exists():
    raise RuntimeError('3번 셀을 먼저 실행해서 생성 설정 JSON을 만들어주세요.')
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
device = config['device']
dtype = getattr(torch, config['dtype'])
weights = download_weights()

qwen_tokenizer, t5_tokenizer = load_tokenizers()
text_encoder = load_text_encoder(weights['text_encoder'], device, dtype)
positive_raw, positive_t5 = encode_prompt(text_encoder, qwen_tokenizer, t5_tokenizer, config['prompt'], device, dtype)
negative_raw, negative_t5 = encode_prompt(text_encoder, qwen_tokenizer, t5_tokenizer, config['negative_prompt'], device, dtype)

CONDITIONING_PATH = Path('/content/anima_conditioning.pt')
torch.save({
    'positive_raw': positive_raw.cpu(),
    'positive_t5': positive_t5.cpu(),
    'negative_raw': negative_raw.cpu(),
    'negative_t5': negative_t5.cpu(),
}, CONDITIONING_PATH)

del text_encoder, positive_raw, positive_t5, negative_raw, negative_t5
gc.collect(); torch.cuda.empty_cache()
print('conditioning:', CONDITIONING_PATH)


In [ ]:
# 6. Anima text adapter + DiT + CFG + Euler -> latent 저장
import gc, json, torch
from pathlib import Path
from anima_core.runtime import download_weights, load_dit, adapt_conditioning, sample_euler

CONFIG_PATH = Path('/content/anima_minimal_config.json')
CONDITIONING_PATH = Path('/content/anima_conditioning.pt')
if not CONFIG_PATH.exists():
    raise RuntimeError('3번 셀을 먼저 실행해서 생성 설정 JSON을 만들어주세요.')
if not CONDITIONING_PATH.exists():
    raise RuntimeError('5번 셀을 먼저 실행해서 conditioning 파일을 만들어주세요.')

config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
device = config['device']
dtype = getattr(torch, config['dtype'])
weights = download_weights()
conditioning = torch.load(CONDITIONING_PATH, map_location='cpu')

dit = load_dit(weights['dit'], device, dtype)
positive = adapt_conditioning(
    dit,
    conditioning['positive_raw'].to(device=device, dtype=dtype),
    conditioning['positive_t5'].to(device),
)
negative = adapt_conditioning(
    dit,
    conditioning['negative_raw'].to(device=device, dtype=dtype),
    conditioning['negative_t5'].to(device),
)

latents = sample_euler(
    dit,
    positive,
    negative,
    config['height'],
    config['width'],
    config['seed'],
    config['steps'],
    config['cfg'],
    denoise=config['denoise'],
    shift=config['shift'],
    device=device,
    dtype=dtype,
)

LATENTS_PATH = Path('/content/anima_latents.pt')
torch.save(latents.cpu(), LATENTS_PATH)
del dit, positive, negative, latents, conditioning
gc.collect(); torch.cuda.empty_cache()
print('latents:', LATENTS_PATH)


In [ ]:
# 7. VAE decode -> PNG
import gc, json, torch
from pathlib import Path
from IPython.display import display
from anima_core.runtime import download_weights, load_vae, decode_image

CONFIG_PATH = Path('/content/anima_minimal_config.json')
LATENTS_PATH = Path('/content/anima_latents.pt')
if not CONFIG_PATH.exists():
    raise RuntimeError('3번 셀을 먼저 실행해서 생성 설정 JSON을 만들어주세요.')
if not LATENTS_PATH.exists():
    raise RuntimeError('6번 셀을 먼저 실행해서 latent 파일을 만들어주세요.')

config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
device = config['device']
dtype = getattr(torch, config['dtype'])
weights = download_weights()
latents = torch.load(LATENTS_PATH, map_location='cpu').to(device=device, dtype=dtype)

vae = load_vae(weights['vae'], device, dtype)
image = decode_image(vae, latents, device)
OUTPUT = Path('/content/anima_minimal.png')
image.save(OUTPUT)
display(image)
print(OUTPUT)

del vae, latents
gc.collect(); torch.cuda.empty_cache()
